# BioSentinel — DNABERT-2 on Kaggle GPU (3 samples × 250K)

## Setup checklist (do these BEFORE running)

1. **Upload FASTAs as a Kaggle Dataset:**
   - Go to https://www.kaggle.com/datasets → **+ New Dataset**
   - Title: `biosentinel-fastas-250k`
   - Upload these **6 files** (2 per sample):
     - `SRR37006657_classified_for_embedding.fasta`
     - `SRR37006657_unclassified_250k.fasta`
     - `SRR37006671_classified_for_embedding.fasta`
     - `SRR37006671_unclassified_250k.fasta`
     - `<SAMPLE3>_classified_for_embedding.fasta`
     - `<SAMPLE3>_unclassified_250k.fasta`
   - Set visibility: **Private**
   - Click **Create**

2. **Open this notebook in Kaggle:**
   - https://www.kaggle.com/code → **+ New Notebook** → Import → upload this `.ipynb`

3. **Configure the notebook:**
   - Right sidebar → **Settings** (gear icon)
   - **Accelerator: GPU P100** (T4 x2 also fine)
   - **Internet: ON** (needed to download DNABERT-2 weights)
   - **Add data** → search `biosentinel-fastas-250k` → **+ Add**

4. **EDIT the `SAMPLES` list in Step 3** to include your sample 3 accession.

5. **Run All** — takes ~2 hrs total (3 samples × ~40 min each). Save Version → Commit when done.

## Step 1 — Verify GPU and check input data

In [11]:
import torch
import os

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('Enable GPU in notebook Settings → Accelerator before running.')

INPUT_DIR = '/kaggle/input/datasets/divyasitani/dataset-v3'
print(f'\nInput files in {INPUT_DIR}:')
for f in sorted(os.listdir(INPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(INPUT_DIR, f)) / (1024 * 1024)
    print(f'  {f}  ({size_mb:.1f} MB)')

CUDA available: True
GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB

Input files in /kaggle/input/datasets/divyasitani/dataset-v3:
  SRR37006657_classified_for_embedding.fasta  (8.3 MB)
  SRR37006657_unclassified_250k.fasta  (41.2 MB)
  SRR37006671_classified_for_embedding.fasta  (8.3 MB)
  SRR37006671_unclassified_250k.fasta  (41.0 MB)


## Step 2 — Install dependencies

Pin transformers to a version that works with DNABERT-2 + has prebuilt wheels.

In [13]:
!pip install -q transformers==4.29.2 tokenizers==0.13.3 einops accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 16.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 11.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


**RESTART the kernel after this cell:** Run → Restart & Run All from this point.

(Old transformers stays in memory until kernel restart. Skipping this causes import errors.)

## Step 3 — Configuration

**EDIT `SAMPLES` to match what's in your Kaggle dataset.** The third entry is a placeholder — replace it with your actual sample 3 accession/label.

In [ ]:
SAMPLES = [
    'SRR37006657',
    'SRR37006671',
    'SAMPLE3',          # <-- REPLACE with your sample 3 accession
]

BATCH_SIZE   = 64
MAX_READ_LEN = 512
MODEL_NAME   = 'zhihan1996/DNABERT-2-117M'

INPUT_DIR  = '/kaggle/input/biosentinel-fastas-250k'
OUTPUT_DIR = '/kaggle/working'

import os
missing = []
for acc in SAMPLES:
    cl  = f'{INPUT_DIR}/{acc}_classified_for_embedding.fasta'
    ucl = f'{INPUT_DIR}/{acc}_unclassified_250k.fasta'
    if not os.path.exists(cl):  missing.append(cl)
    if not os.path.exists(ucl): missing.append(ucl)
if missing:
    print('MISSING FILES:')
    for m in missing:
        print(f'  {m}')
    raise FileNotFoundError('Fix SAMPLES list or upload missing files to the Kaggle dataset.')
print(f'All {len(SAMPLES)*2} input files present.')

## Step 4 — Load DNABERT-2 (once, reused across samples)

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModel

device = torch.device('cuda')

print('Loading DNABERT-2...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval().to(device)
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M')

## Step 5 — Helper functions

In [ ]:
import numpy as np
from tqdm.auto import tqdm

def read_fasta(path):
    print(f'  Reading {path}...')
    records = []
    current_id, current_seq = None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id is not None:
                    records.append((current_id, ''.join(current_seq)))
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
    if current_id is not None:
        records.append((current_id, ''.join(current_seq)))
    print(f'  Loaded {len(records):,} sequences')
    return records


def embed_sequences(sequences, batch_size, max_len, desc='Embedding'):
    all_embeddings = []
    for i in tqdm(range(0, len(sequences), batch_size), desc=desc):
        batch = sequences[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_len,
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        hidden = outputs[0] if isinstance(outputs, tuple) else outputs.last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        sum_emb = (hidden * mask).sum(dim=1)
        count = mask.sum(dim=1).clamp(min=1e-9)
        mean_emb = sum_emb / count
        all_embeddings.append(mean_emb.cpu().float().numpy())
    return np.vstack(all_embeddings)


def process_sample(acc):
    """Embed both classified + unclassified for one sample, save to /kaggle/working."""
    print(f'\n{"="*60}\nSample: {acc}\n{"="*60}')
    out = {}

    for kind, fname_suffix in [('classified',   'classified_for_embedding.fasta'),
                                ('unclassified', 'unclassified_250k.fasta')]:
        in_path = f'{INPUT_DIR}/{acc}_{fname_suffix}'
        records = read_fasta(in_path)
        ids, seqs = zip(*records)

        t0 = time.time()
        emb = embed_sequences(list(seqs), BATCH_SIZE, MAX_READ_LEN, f'{acc}-{kind}')
        print(f'  Embedded {len(emb):,} {kind} reads in {(time.time()-t0)/60:.1f} min')

        np.save(f'{OUTPUT_DIR}/{acc}_{kind}_embeddings.npy', emb)
        with open(f'{OUTPUT_DIR}/{acc}_{kind}_ids.txt', 'w') as f:
            f.write('\n'.join(ids))
        out[kind] = emb
        print(f'  Saved {acc}_{kind}_embeddings.npy and _ids.txt')
    return out

## Step 6 — Embed all samples (~2 hrs total on P100)

In [ ]:
all_embeddings = {}
session_start = time.time()
for acc in SAMPLES:
    all_embeddings[acc] = process_sample(acc)
print(f'\nAll {len(SAMPLES)} samples done in {(time.time()-session_start)/60:.1f} min')

## Step 7 — Sanity check

Embeddings should be **continuous** (not collapsed). Healthy values:
- Mean std across dims: > 0.05
- Unique vector norms: many thousands

In [ ]:
from numpy.linalg import norm

for acc in SAMPLES:
    print(f'\n=== {acc} ===')
    for kind in ['classified', 'unclassified']:
        emb = all_embeddings[acc][kind]
        norms = norm(emb, axis=1)
        n_unique = len(np.unique(np.round(norms, 4)))
        print(f'  {kind:13} shape={emb.shape}  '
              f'std-of-dims={emb.std(axis=0).mean():.4f}  '
              f'unique-norms={n_unique:,}/{len(norms):,}')
        if n_unique < 1000:
            print(f'    ✗ WARNING: collapsed embeddings for {acc} {kind}')

## Step 8 — Commit & download

1. Click **Save Version** (top right) → **Save & Run All (Commit)**
2. Wait for the commit to finish (it re-runs the notebook in batch)
3. Once committed, go to the **Output** tab
4. Download all 12 files (4 per sample × 3 samples):
   - `<ACC>_classified_embeddings.npy`
   - `<ACC>_classified_ids.txt`
   - `<ACC>_unclassified_embeddings.npy`
   - `<ACC>_unclassified_ids.txt`
5. Place them in your local `embeddings/` folder
6. Run locally: `python anomaly_detection_v2.py` (updated for 3 samples)

In [ ]:
import os
print('Files in /kaggle/working (downloadable from Output tab):')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(f'{OUTPUT_DIR}/{f}') / (1024 * 1024)
    print(f'  {f}  ({size_mb:.1f} MB)')